# Harmonized data: checks
Run after `uv run python scripts/harmonize.py`. See README.md, "Harmonization".

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import ppg_bp_database as db
from ppg_bp_database.utils import HARMONIZED_DIR

CONFIG = "default"
windows = db.load_windows(CONFIG)
bp = db.load_window_bp(CONFIG)

## Kept / rejected windows per dataset and reason
High rejection rates for one reason -> look at examples, maybe tune the threshold in `harmonize/config.py`.

In [ ]:
report = pd.read_csv(HARMONIZED_DIR / CONFIG / "report.csv")
report.pivot(index="reason", columns="dataset", values="count").fillna(0).astype(int)

## PPG polarity
`ppg_dskew` > 0 = steep upstroke, slow decay (correct). A dataset with mostly negative values is inverted: set `ppg_inverted` in `harmonize/sources.py`.

In [ ]:
windows.groupby("dataset")["ppg_dskew"].describe()[["count", "25%", "50%", "75%"]].assign(
    positive=windows.groupby("dataset")["ppg_dskew"].apply(lambda s: (s > 0).mean()))

## Random windows per dataset

In [ ]:
datasets = sorted(windows["dataset"].unique())
fig, axes = plt.subplots(len(datasets), 1, figsize=(12, 2.5 * len(datasets)), squeeze=False)
for ax, name in zip(axes[:, 0], datasets):
    w = windows[windows["dataset"] == name].sample(1, random_state=0)
    arrays = db.load_window_waves(w, CONFIG)
    t = np.arange(arrays["ppg"].shape[1]) / 125
    ax.plot(t, arrays["ppg"][0], label="ppg (z)")
    if w["has_abp"].iloc[0]:
        ax2 = ax.twinx()
        ax2.plot(t, arrays["abp"][0], color="C1", alpha=0.6, label="abp (mmHg)")
        rows = bp[bp["window_id"] == w["window_id"].iloc[0]]
        ax2.plot(rows["time_s"], rows["dbp"], "kx")  # beat onsets
    ax.set_title(f"{w['window_id'].iloc[0]} ({w['condition'].iloc[0]})")
fig.tight_layout()

## BP distributions per dataset (median per window)

In [ ]:
per_window = bp.groupby(["dataset", "window_id"])[["sbp", "dbp"]].median().reset_index()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col in zip(axes, ["sbp", "dbp"]):
    for name, g in per_window.groupby("dataset"):
        ax.hist(g[col], bins=60, histtype="step", density=True, label=name)
    ax.set_xlabel(f"{col} (mmHg)")
axes[0].legend()

## MIMIC-BP: our beats vs the dataset's own 30 s medians
The dataset's labels stay in `data/structured`; per record, the median of our harmonized beats should match them closely.

In [ ]:
if "mimicbp" in windows["dataset"].unique():
    given = db.load_bp("mimicbp")
    given = given[given["source"] == "dataset"].set_index("record_id")[["sbp", "dbp"]]
    ours = bp[bp["dataset"] == "mimicbp"].merge(windows[["window_id", "record_id"]], on="window_id")
    ours = ours.groupby("record_id")[["sbp", "dbp"]].median()
    diff = (ours - given.loc[ours.index])
    print(diff.describe().round(2))